# Melbourne Housing Price Analysis

An in-depth exploratory analysis of Melbourne, Australia's real estate market — examining price distributions, suburb-level patterns, property characteristics, and the factors that drive housing prices.

## Project Overview

Melbourne's housing market is one of the most dynamic in Australia. This notebook analyzes real estate transaction data to understand:

- How property prices are distributed across the city
- What factors (rooms, location, land size, property type) influence prices
- Suburb and regional price differences
- Relationships between property features and sale prices
- Market timing patterns

## Learning Objectives

1. Analyze real estate data with mixed numerical and categorical features
2. Handle missing values strategically in property datasets
3. Perform geographic-aware analysis using suburb and region data
4. Create visualizations for price distributions and relationships
5. Apply statistical tests to validate pricing hypotheses
6. Extract actionable insights from housing market data

## Business / Research Problem

**Core question:** What factors most strongly influence Melbourne property prices, and how do prices vary across regions and property types?

This analysis benefits home buyers, real estate agents, urban planners, and investors looking to understand Melbourne's property landscape.

## Why This Analysis Matters

- **Buyer guidance:** Understanding price drivers helps buyers make informed decisions
- **Investment analysis:** Identifies potentially undervalued regions
- **Urban planning:** Reveals spatial inequality and development patterns
- **Data science practice:** Rich dataset with geographic, temporal, and numerical features

## Dataset Overview

| Column | Description |
|--------|-------------|
| Suburb | Suburb name |
| Address | Property address |
| Rooms | Number of rooms |
| Type | h = house, u = unit, t = townhouse |
| Price | Sale price (AUD) |
| Method | S = sold, SP = sold prior, PI = passed in, etc. |
| SellerG | Real estate agent |
| Date | Sale date |
| Distance | Distance from CBD (km) |
| Postcode | Postal code |
| Bedroom2 | Number of bedrooms (from different source) |
| Bathroom | Number of bathrooms |
| Car | Number of car spots |
| Landsize | Land area (sqm) |
| BuildingArea | Building area (sqm) |
| YearBuilt | Year property was built |
| CouncilArea | Local council area |
| Lattitude | Latitude |
| Longtitude | Longitude |
| Regionname | General region |
| Propertycount | Number of properties in suburb |

## Dataset Source and License Notes

- **Source:** [Kaggle — Melbourne Housing Snapshot](https://www.kaggle.com/datasets/dansbecker/melbourne-housing-snapshot)
- **License:** CC BY-NC-SA 4.0
- **Origin:** Scraped from publicly available real estate listings
- **Period:** 2016-2018

## Environment Setup

In [ ]:
!pip install -q pandas numpy matplotlib seaborn scipy kaggle

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("All imports loaded.")

## Configuration / Constants

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = 'data'

## Dataset Download

In [ ]:
import os
os.makedirs(DATA_DIR, exist_ok=True)

!kaggle datasets download -d dansbecker/melbourne-housing-snapshot -p {DATA_DIR} --unzip --force

# Find data file
data_file = None
for f in os.listdir(DATA_DIR):
    if f.endswith('.csv'):
        data_file = os.path.join(DATA_DIR, f)
        break

if data_file is None and os.path.exists('data.csv'):
    data_file = 'data.csv'

print(f"Data file: {data_file}")

## Data Loading

In [ ]:
df = pd.read_csv(data_file)
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## Data Validation Checks

In [ ]:
print("=== Data Types ===")
print(df.dtypes)

print("\n=== Missing Values ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_info = pd.DataFrame({'Count': missing, '%': missing_pct})
print(missing_info[missing_info['Count'] > 0].sort_values('%', ascending=False))

print(f"\nDuplicates: {df.duplicated().sum()}")
print(f"\nPrice range: ${df['Price'].min():,.0f} — ${df['Price'].max():,.0f}")
print(f"Suburbs: {df['Suburb'].nunique()}")
print(f"Property types: {df['Type'].unique()}")
print(f"Regions: {df['Regionname'].nunique()}")

## Data Cleaning

In [ ]:
# Drop rows with missing Price (our target variable)
before = len(df)
df = df.dropna(subset=['Price'])
print(f"Dropped {before - len(df)} rows with missing Price")

# Parse dates
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['SaleYear'] = df['Date'].dt.year
df['SaleMonth'] = df['Date'].dt.month

# Map property types to readable names
type_map = {'h': 'House', 'u': 'Unit', 't': 'Townhouse'}
df['PropertyType'] = df['Type'].map(type_map)

# Property age (approximate)
df['PropertyAge'] = df['SaleYear'] - df['YearBuilt']

# Price per room
df['PricePerRoom'] = df['Price'] / df['Rooms'].clip(lower=1)

# Remove obvious outliers
df = df[(df['Landsize'] < 50000) | (df['Landsize'].isna())]  # Remove extreme land sizes
df = df[df['Price'] > 50000]  # Remove suspiciously cheap properties

print(f"\nFinal shape: {df.shape}")
print(f"New features: SaleYear, SaleMonth, PropertyType, PropertyAge, PricePerRoom")

## Exploratory Data Analysis

### Price Distribution

In [ ]:
print("=== Price Statistics ===")
print(f"Mean:   ${df['Price'].mean():>12,.0f}")
print(f"Median: ${df['Price'].median():>12,.0f}")
print(f"Std:    ${df['Price'].std():>12,.0f}")
print(f"IQR:    ${df['Price'].quantile(0.25):>12,.0f} — ${df['Price'].quantile(0.75):,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Price']/1e6, bins=50, edgecolor='black', alpha=0.7, color='#3498db')
axes[0].axvline(df['Price'].median()/1e6, color='red', ls='--', label=f"Median: ${df['Price'].median()/1e6:.2f}M")
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (AUD Millions)')
axes[0].legend()

axes[1].hist(np.log10(df['Price']), bins=40, edgecolor='black', alpha=0.7, color='#2ecc71')
axes[1].set_title('Log₁₀ Price Distribution')
axes[1].set_xlabel('log₁₀(Price)')

plt.tight_layout()
plt.show()

## Univariate Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Rooms
df['Rooms'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], color='#3498db')
axes[0,0].set_title('Number of Rooms')

# Property Type
df['PropertyType'].value_counts().plot(kind='bar', ax=axes[0,1], color='#e74c3c')
axes[0,1].set_title('Property Type')
axes[0,1].tick_params(axis='x', rotation=0)

# Distance from CBD
axes[0,2].hist(df['Distance'].dropna(), bins=30, edgecolor='black', alpha=0.7, color='#2ecc71')
axes[0,2].set_title('Distance from CBD')
axes[0,2].set_xlabel('km')

# Region
df['Regionname'].value_counts().plot(kind='barh', ax=axes[1,0], color='#9b59b6')
axes[1,0].set_title('Region Distribution')
axes[1,0].invert_yaxis()

# Bathroom
df['Bathroom'].value_counts().sort_index().head(8).plot(kind='bar', ax=axes[1,1], color='#f1c40f')
axes[1,1].set_title('Bathrooms')

# Sale method
df['Method'].value_counts().plot(kind='bar', ax=axes[1,2], color='#e67e22')
axes[1,2].set_title('Sale Method')
axes[1,2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Bivariate / Multivariate Analysis

### Price vs Key Features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price by property type
sns.boxplot(data=df, x='PropertyType', y='Price', ax=axes[0,0], palette='Set2')
axes[0,0].set_title('Price by Property Type')
axes[0,0].set_ylim(0, df['Price'].quantile(0.95))

# Price by rooms
room_price = df.groupby('Rooms')['Price'].median()
room_price[room_price.index <= 6].plot(kind='bar', ax=axes[0,1], color='#3498db')
axes[0,1].set_title('Median Price by Rooms')
axes[0,1].set_ylabel('Median Price (AUD)')

# Price vs Distance
axes[1,0].scatter(df['Distance'], df['Price']/1e6, alpha=0.1, s=5, color='#e74c3c')
axes[1,0].set_xlabel('Distance from CBD (km)')
axes[1,0].set_ylabel('Price (M AUD)')
axes[1,0].set_title('Price vs Distance from CBD')
axes[1,0].set_ylim(0, 5)

# Price by region
region_price = df.groupby('Regionname')['Price'].median().sort_values()
region_price.plot(kind='barh', ax=axes[1,1], color='#2ecc71')
axes[1,1].set_title('Median Price by Region')
axes[1,1].set_xlabel('Median Price (AUD)')

plt.tight_layout()
plt.show()

In [ ]:
# Price by region and property type
pivot = df.pivot_table(values='Price', index='Regionname', columns='PropertyType', aggfunc='median')

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot/1e6, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Median Price ($ Millions) by Region × Type')
plt.tight_layout()
plt.show()

## Feature-Specific Insights

In [ ]:
# Top 15 most expensive suburbs
suburb_stats = df.groupby('Suburb').agg(
    Count=('Price', 'count'),
    Median_Price=('Price', 'median'),
    Avg_Distance=('Distance', 'mean')
).query('Count >= 10').sort_values('Median_Price', ascending=False)

print("Top 15 Most Expensive Suburbs (min 10 sales):")
print(suburb_stats.head(15).round(0).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
top15 = suburb_stats.head(15)
ax.barh(range(len(top15)), top15['Median_Price']/1e6, color='#f1c40f')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15.index)
ax.invert_yaxis()
ax.set_title('Top 15 Most Expensive Suburbs (Median Price)')
ax.set_xlabel('Median Price (AUD M)')
plt.tight_layout()
plt.show()

In [ ]:
# Seller performance
seller_stats = df.groupby('SellerG').agg(
    Sales=('Price', 'count'),
    Avg_Price=('Price', 'mean'),
    Total_Revenue=('Price', 'sum')
).sort_values('Sales', ascending=False)

print("Top 10 Real Estate Agents by Sales Count:")
print(seller_stats.head(10).round(0).to_string())

# Monthly price trends
if 'Date' in df.columns:
    monthly = df.groupby(df['Date'].dt.to_period('M'))['Price'].agg(['median', 'count'])
    
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(monthly.index.astype(str), monthly['median']/1e6, marker='o', linewidth=2, color='#3498db')
    ax.set_title('Monthly Median Price Trend')
    ax.set_ylabel('Median Price (AUD M)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## Statistical Checks / Hypothesis Testing

In [ ]:
alpha = 0.05

# 1. House vs Unit price difference
house_prices = df[df['Type'] == 'h']['Price']
unit_prices = df[df['Type'] == 'u']['Price']
u, p = stats.mannwhitneyu(house_prices, unit_prices)
print(f"1. House vs Unit price: U={u:.0f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")
print(f"   House median: ${house_prices.median():,.0f}, Unit median: ${unit_prices.median():,.0f}")

# 2. Distance-Price correlation
valid = df[['Distance', 'Price']].dropna()
r, p = stats.pearsonr(valid['Distance'], valid['Price'])
print(f"\n2. Distance-Price correlation: r={r:.3f}, p={p:.2e}")

# 3. Regional price differences
groups = [g['Price'].values for _, g in df.groupby('Regionname')]
h, p = stats.kruskal(*groups)
print(f"\n3. Regional price differences: H={h:.2f}, p={p:.2e} → {'SIGNIFICANT' if p < alpha else 'Not significant'}")

# 4. Rooms-Price correlation
r, p = stats.spearmanr(df['Rooms'], df['Price'])
print(f"\n4. Rooms-Price (Spearman): ρ={r:.3f}, p={p:.2e}")

# 5. Bathroom-Price correlation
valid = df[['Bathroom', 'Price']].dropna()
r, p = stats.spearmanr(valid['Bathroom'], valid['Price'])
print(f"5. Bathroom-Price (Spearman): ρ={r:.3f}, p={p:.2e}")

## Visual Analysis

In [ ]:
# Correlation heatmap
num_cols = ['Price', 'Rooms', 'Distance', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'Propertycount']
available = [c for c in num_cols if c in df.columns]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[available].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Geographic scatter plot
valid = df.dropna(subset=['Lattitude', 'Longtitude'])

fig, ax = plt.subplots(figsize=(10, 10))
scatter = ax.scatter(valid['Longtitude'], valid['Lattitude'],
                     c=valid['Price']/1e6, cmap='YlOrRd', s=3, alpha=0.5)
plt.colorbar(scatter, label='Price (AUD M)', ax=ax)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Melbourne Property Prices — Geographic Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# Price distribution by region
fig, ax = plt.subplots(figsize=(14, 6))
region_order = df.groupby('Regionname')['Price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Regionname', y='Price', order=region_order, ax=ax, palette='Set2')
ax.set_ylim(0, df['Price'].quantile(0.95))
ax.set_title('Price Distribution by Region')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Key Findings

1. **Strong distance effect** — Properties closer to the CBD command significantly higher prices
2. **Houses dominate** — Houses are the most common property type and typically most expensive
3. **Regional variation** — Southern Metropolitan and inner suburbs are the most expensive regions
4. **Room count matters** — More rooms correlate with higher prices, but with diminishing returns
5. **Price distribution is right-skewed** — Most properties cluster around the median, with a long tail of luxury homes
6. **Property type premium** — Houses command a significant premium over units, even controlling for size
7. **Geographic clustering** — High-value properties cluster along specific corridors and bayside areas

## Limitations

- **Missing data** — BuildingArea (~52%) and YearBuilt (~42%) have significant missing values
- **Time period** — Data covers 2016-2018; market conditions have changed since
- **No interior details** — Renovation quality, condition, and finishes not captured
- **Self-reported area** — Landsize and building area may have measurement inconsistencies
- **Outliers** — Extreme prices and land sizes may represent data errors or special cases

## Recommendations / Next Steps

1. **Price prediction model** — Build regression models to predict property prices
2. **Feature engineering** — Create interaction features (rooms × distance, etc.)
3. **Temporal analysis** — Study seasonal patterns in prices and sales volumes
4. **Merge external data** — Add school zones, public transport access, crime data
5. **Interactive maps** — Use Folium or Plotly for interactive geographic visualization

### How to Extend This Analysis

- Scrape current listings to compare with historical data and identify appreciation rates
- Analyze the impact of specific council policies on property values
- Build a suburb recommendation engine based on buyer preferences and budget

## Common Mistakes

1. **Not removing price outliers** — Extreme values (e.g., $1 or $100M) distort statistics
2. **Using mean for skewed prices** — Median is a better measure of central tendency for prices
3. **Ignoring missing BuildingArea** — 50%+ is missing; don't drop all those rows — impute or work around
4. **Treating PostCode as numeric** — It's a categorical identifier, not a numerical feature
5. **Not considering multicollinearity** — Rooms, Bedroom2, and Bathroom are correlated
6. **Confusing correlation with causation** — Distance doesn't "cause" price; it's a proxy for location quality

## Mini Challenge / Exercises

1. **Best value suburbs:** Find suburbs where the median price-per-room is lowest but are still within 15km of CBD.

2. **Price premium:** Calculate the average premium for each additional bathroom, holding rooms constant.

3. **Old vs new:** Compare median prices for properties built before 1950 vs after 2000.

4. **Seller specialization:** Which real estate agents specialize in high-end properties (top quartile by price)?

5. **Distance bands:** Create 5km distance bands from CBD and show how median price, rooms, and land size change.

In [ ]:
# Exercise space
# Exercise 5: Distance bands
# df['DistBand'] = pd.cut(df['Distance'], bins=[0, 5, 10, 15, 20, 30, 50],
#                         labels=['0-5km', '5-10km', '10-15km', '15-20km', '20-30km', '30-50km'])
# print(df.groupby('DistBand', observed=False)['Price'].median().to_string())

print("Uncomment the exercises above and try them!")

## Final Summary / Key Takeaways

| Factor | Impact on Price |
|--------|----------------|
| Distance from CBD | Strong negative correlation |
| Property Type | Houses > Townhouses > Units |
| Rooms | Positive correlation |
| Region | Southern Metropolitan most expensive |
| Land Size | Positive correlation (for houses) |
| Bathrooms | Significant price premium |

Melbourne's property market shows clear spatial patterns — proximity to the CBD is the dominant price driver, followed by property type and size. The market is highly segmented by region, with inner suburbs commanding 2-3x the price of outer suburbs. Understanding these patterns provides valuable context for buyers, investors, and urban planners seeking to navigate Melbourne's dynamic real estate landscape.